# Train FOMO 160x160, lưới 20x20 (car / human) - không dùng Edge Impulse

Notebook này huấn luyện model **FOMO** (Faster Objects, More Objects) nhận diện tâm **xe** và **người** trên ảnh xám
160x160 (lưới đầu ra 20x20, mỗi ô 8 px), rồi xuất `.tflite` int8 cho firmware `smart-street-lighting`
(ESP32-S3, TFLite Micro + ESP-NN).

**FOMO là gì, vì sao dùng nó?** Model detection thường (YOLO) dự đoán hộp bao, quá nặng cho ESP32-S3.
FOMO bỏ hộp, chỉ trả lời câu hỏi "ô 8x8 này có tâm vật thể lớp nào?" trên lưới 20x20. Đèn đường chỉ cần **tâm vật**
để biết vật ở zone nào (4 zone = 4 khoảng toạ độ x) và đang đi hay đứng yên, nên FOMO đủ dùng mà nhẹ hơn hàng chục lần.

**Vì sao 160x160 và lưới 20x20:**
- Ảnh gốc 320x240, vật trung vị khoảng 24x20 px. Thu về 160 thì vật còn khoảng 12x13 px; ở 96 thì chỉ còn khoảng 7 px, dưới ngưỡng 8-10 px của kế hoạch.
- Cắt MobileNetV2 ở `block_6_expand_relu` (độ phân giải 1/8) cho đúng 160/8 = 20, mỗi ô 8 px.
- Đo trên dataset: lưới 20x20 chỉ gộp **0.4%** số vật vào chung ô (lưới 10x10 gộp khoảng 5%).
- Mỗi zone rộng 5 ô.

**Dữ liệu**: export **COCO** từ Roboflow (`HTN.v2i.coco`): thư mục `train/ valid/ test/`, mỗi thư mục có
`_annotations.coco.json`. Roboflow đã tạo 3 bản augment cho mỗi ảnh train (độ sáng ±18%, blur, nhiễu muối tiêu).
Ảnh có 2 nguồn: **320x240** (camera OV2640) và **640x360**. Điểm đánh giá được tách riêng hai nguồn.

**Tiền xử lý phải giống firmware**: ảnh xám, **ép cả khung** (không giữ tỉ lệ, không cắt) về 160x160 bằng
**trung bình diện tích** (`cv2.INTER_AREA`), không cân bằng histogram. Notebook không cắt ô vuông giữa như bản 96
của dự án đếm xe, vì **hai mép ảnh là zone 1 và zone 4**. Firmware phải cắt ROI giống lúc thu dataset rồi resize y hệt.

**Luồng chạy:** cấu hình -> nạp dữ liệu + kiểm tra rò rỉ -> nhãn FOMO -> augmentation -> model -> train ->
đánh giá trên val (chọn ngưỡng) -> test một lần -> int8 -> so float vs int8 -> xuất file.

## Cách chạy

**Kaggle** (khuyên dùng):
1. *Create -> New Dataset*, tải lên `HTN.v2i.coco.zip` (trong `E:\TKHTN\dataset\`). Kaggle tự giải nén.
2. Tạo Notebook, *Add Input* -> chọn dataset đó, **bật Internet** (để tải trọng số ImageNet), bật GPU (T4/P100), rồi *Run All*.
3. Notebook tự tìm `train/_annotations.coco.json` trong `/kaggle/input`. Nếu muốn tải qua `kagglehub`, điền `KAGGLE_DATASET = "owner/slug"` ở ô cấu hình.

**Colab**: tải `HTN.v2i.coco.zip` lên Drive `MyDrive/streetlight_training/`, rồi Run all (notebook sẽ hỏi quyền Drive).

**Chạy thử nhanh** (chỉ để kiểm tra notebook không lỗi, kết quả vô nghĩa): đặt `QUICK_RUN = True` ở ô cấu hình.

Tập **test chỉ xem một lần ở cuối**. Ngưỡng được chọn trên **val**.

## 1. Cấu hình

- `CELL = 8` nên `GRID = 160 / 8 = 20`. `CUT_LAYER` được tự chọn: ô 8 px tương ứng `block_6_expand_relu` (độ phân giải 1/8).
- `OBJECT_WEIGHT`:
  - Trong 400 ô chỉ có vài ô có vật. Không đặt trọng số thì model học "luôn đoán nền" là đã đúng khoảng 99%.
  - Edge Impulse dùng 100 cho **1 ô/vật**.
  - Ở đây vật nhỏ (khoảng 1.5 ô), lõi đo được chỉ **khoảng 1.03 ô/vật** (in ra ở mục 3), nên giữ **100** như Edge Impulse
    để tổng trọng số mỗi vật tương đương. (Bản 96 của dự án đếm xe hạ còn 25 vì xe ở đó có lõi 2-9 ô.)
  - Nếu model đoán "có vật" quá tay (nhiều FP loại `background`), thử giảm xuống 50.
- `MATCH_RADIUS_PX = 12`: dự đoán là TP nếu tâm cách tâm nhãn ≤ 12 px (1.5 ô). Tính bằng pixel để so được giữa các cỡ lưới.
- `CALIBRATION_IMAGES`: số ảnh **train** để hiệu chỉnh int8. Bản 96 của dự án đếm xe từng thấy dùng 400 ảnh làm tụt F1, nên dùng 2000.
- `EPOCHS_HEAD` / `EPOCHS_FINETUNE`: pha 1 chỉ học phần đầu mới (backbone đông cứng), pha 2 học cả backbone với
  tốc độ học nhỏ, tránh phá trọng số ImageNet ngay từ đầu.

In [ ]:
IMG_SIZE = 160                      # model input: whole grey frame squashed to 160x160 (INTER_AREA)
CELL = 8                            # one output cell covers CELL x CELL input pixels
GRID = IMG_SIZE // CELL             # 20x20 output grid
MATCH_RADIUS_PX = 12.0              # TP if the centre is within 12 px of a truth (1.5 cells at CELL 8); kept in
                                    # pixels so models with different grids are scored alike
CLASSES = ["background", "car", "human"]   # index 0 must stay background (firmware relies on it);
                                           # the others must match the COCO category names
OBJECT_CLASSES = CLASSES[1:]

import os

MODEL_STEM = "fomo_160_g20"
ALPHA = 0.35                         # MobileNetV2 width (Edge Impulse FOMO uses 0.35)
# 1/8 resolution (20x20 at 160) or, for 16 px cells, 1/16 (10x10 at 160);
# any other cell size (e.g. 10 px) is resampled from the 1/8 features in build_model
CUT_LAYER = "block_13_expand_relu" if CELL % 16 == 0 else "block_6_expand_relu"
HEAD_FILTERS = 32
OBJECT_WEIGHT = 100.0                # per core cell; cores are ~1 cell per object here (see encode_targets)
BATCH_SIZE = 32
EPOCHS_HEAD = 15
EPOCHS_FINETUNE = 60
LEARNING_RATE = 1e-3
FINETUNE_LR = 2e-4
PATIENCE = 12                        # epochs without a better val macro F1 before stopping
EVAL_THRESHOLD = 0.5                 # threshold used while training to track val F1
CALIBRATION_IMAGES = 2000            # train images used to calibrate int8 quantisation (never val/test)
SEED = 42
KAGGLE_DATASET = ""                  # optional "owner/slug" downloaded with kagglehub; empty = search inputs
SPLIT_FOLDERS = {"train": "train", "val": "valid", "test": "test"}   # Roboflow folder names
DATA_DIR = os.environ.get("DATA_DIR", "")
WORK_DIR = os.environ.get("WORK_DIR", "")
QUICK_RUN = os.environ.get("QUICK_RUN", "0") == "1"

In [ ]:
import datetime
import hashlib
import json
import math
import shutil
import sys
import zipfile
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

keras.utils.set_random_seed(SEED)
print("TensorFlow", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "none (CPU works, slower)")
if QUICK_RUN:
    EPOCHS_HEAD, EPOCHS_FINETUNE, CALIBRATION_IMAGES = 1, 1, 20
    print("QUICK_RUN: smoke test only, results are meaningless")

## 2. Tìm và nạp dữ liệu

Mỗi split là một file COCO: `images` (tên file, kích thước), `annotations` (`bbox = [x, y, w, h]` theo pixel ảnh gốc),
`categories`. Roboflow thêm một category cha (`HTN`, `supercategory = "none"`) không gán cho hộp nào, nên bỏ đi.
Tên các category còn lại **phải khớp** `CLASSES`; nếu có lớp lạ, notebook báo lỗi thay vì âm thầm bỏ nhãn.

Mỗi hộp được đổi sang toạ độ trên ảnh 160x160: `(class, cx, cy, w, h)`. Ảnh nào không có hộp vẫn được giữ lại,
vì chúng dạy model rằng đường trống thì không có gì.

In [ ]:
def work_dir() -> Path:
    # Writable scratch folder: WORK_DIR, Kaggle, Colab, or ./work next to the notebook.
    if WORK_DIR:
        return Path(WORK_DIR)
    for candidate in ("/kaggle/working", "/content"):
        if Path(candidate).exists():
            return Path(candidate)
    return Path.cwd() / "work"


WORK = work_dir()
OUT_DIR = WORK / "out"
OUT_DIR.mkdir(parents=True, exist_ok=True)


def search_roots() -> list[Path]:
    # Places the COCO export (folder or zip) may be.
    if KAGGLE_DATASET:
        import kagglehub
        return [Path(kagglehub.dataset_download(KAGGLE_DATASET))]
    if "google.colab" in sys.modules:
        try:
            from google.colab import drive
            drive.mount("/content/drive")
        except Exception as error:          # not fatal: the data may already be on local disk
            print("Drive not mounted:", error)
    roots = [DATA_DIR, "/kaggle/input", "/content/drive/MyDrive/streetlight_training", str(WORK / "coco")]
    if not Path("/content").exists() and not Path("/kaggle").exists():
        roots.append(str(Path.cwd()))
    return [Path(r) for r in roots if r and Path(r).exists()]


def find_coco_root(roots: list[Path]) -> Path | None:
    # First folder that holds train/, valid/ and test/ each with _annotations.coco.json.
    for root in roots:
        for ann in sorted(root.rglob("_annotations.coco.json")):
            folder = ann.parent.parent
            if ann.parent.name == SPLIT_FOLDERS["train"] and all(
                    (folder / f / "_annotations.coco.json").exists() for f in SPLIT_FOLDERS.values()):
                return folder
    return None


def locate_coco() -> Path:
    # Folder of the Roboflow COCO export; unzip a *coco*.zip if only the archive is found.
    roots = search_roots()
    found = find_coco_root(roots)
    if found:
        return found
    for root in roots:
        for archive in root.rglob("*coco*.zip"):
            print("unzipping", archive)
            zipfile.ZipFile(archive).extractall(WORK / "coco")
            found = find_coco_root([WORK / "coco"])
            if found:
                return found
    raise FileNotFoundError(f"No Roboflow COCO export (train/valid/test) under {roots}; see 'Cách chạy' above")


COCO_ROOT = locate_coco()
print("dataset:", COCO_ROOT)


def class_map(coco: dict) -> dict[int, int]:
    # COCO category id -> index in CLASSES; refuses unknown or missing names.
    by_name = {c["name"]: c["id"] for c in coco["categories"] if c.get("supercategory") != "none"}
    unknown = sorted(set(by_name) - set(OBJECT_CLASSES))
    missing = sorted(set(OBJECT_CLASSES) - set(by_name))
    if unknown or missing:
        raise ValueError(f"COCO categories {sorted(by_name)} do not match CLASSES {OBJECT_CLASSES}")
    return {by_name[name]: CLASSES.index(name) for name in OBJECT_CLASSES}


def load_split(split: str) -> tuple[np.ndarray, list, pd.DataFrame]:
    # Images (N, IMG_SIZE, IMG_SIZE) uint8, their objects [(class, cx, cy, w, h)] in model pixels, and per-image rows.
    folder = COCO_ROOT / SPLIT_FOLDERS[split]
    coco = json.loads((folder / "_annotations.coco.json").read_text(encoding="utf-8"))
    to_class = class_map(coco)
    boxes: dict[int, list] = {}
    for a in coco["annotations"]:
        boxes.setdefault(a["image_id"], []).append((to_class[a["category_id"]], *map(float, a["bbox"])))
    infos = coco["images"]
    if QUICK_RUN:
        infos = [infos[i] for i in np.random.default_rng(SEED).permutation(len(infos))[:64]]
    images, objects, rows = [], [], []
    for info in infos:
        gray = cv2.imread(str(folder / info["file_name"]), cv2.IMREAD_GRAYSCALE)
        if gray is None:
            raise ValueError(f"cannot read {split}/{info['file_name']}")
        height, width = gray.shape
        sx, sy = IMG_SIZE / width, IMG_SIZE / height
        images.append(cv2.resize(gray, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA))
        objects.append([(cls, (x + w / 2) * sx, (y + h / 2) * sy, w * sx, h * sy)
                        for cls, x, y, w, h in boxes.get(info["id"], []) if w > 0 and h > 0])
        rows.append({"file": info["file_name"], "source": f"{width}x{height}",
                     "source_image": info["file_name"].split(".rf.")[0]})
    return np.stack(images), objects, pd.DataFrame(rows)


X_train, O_train, M_train = load_split("train")
X_val, O_val, M_val = load_split("val")
X_test, O_test, M_test = load_split("test")
SOURCES = sorted(M_train["source"].unique())
MANIFEST_HASH = hashlib.md5("".join(sorted(M_train["file"])).encode()).hexdigest()[:8]
summary = pd.DataFrame({
    s: {"images": len(o), "empty": sum(not x for x in o),
        **{c: sum(1 for x in o for t in x if t[0] == i) for i, c in enumerate(CLASSES) if i},
        **{src: int((m["source"] == src).sum()) for src in SOURCES}}
    for s, o, m in (("train", O_train, M_train), ("val", O_val, M_val), ("test", O_test, M_test))}).T
print(summary)
sizes = np.array([max(t[3], t[4]) for o in O_train for t in o])
print(f"train object size at {IMG_SIZE}px: median {np.median(sizes):.1f} px, "
      f"< 8 px: {np.mean(sizes < 8):.1%}, < 1 cell ({CELL} px): {np.mean(sizes < CELL):.1%}")
if not any(not o for o in O_train):
    print("WARNING: no empty (object-free) training image. Add frames of the empty model street at 20 % and 100 % "
          "lamp level (plan 01, 'Negative') so the model learns that an empty road means no detection.")

### Kiểm tra lại rò rỉ và xem mẫu

Kiểm tra trực tiếp trong notebook, không tin hoàn toàn vào cách chia của Roboflow. Có 2 phép kiểm:
1. **Cùng ảnh nguồn:** Roboflow đặt tên bản augment là `<nguồn>.rf.<hash>.jpg`. Ảnh nguồn nào có mặt ở cả train lẫn val/test là rò rỉ chắc chắn.
2. **Frame gần trùng:** chỉ so nền (sai khác trung bình trên ảnh thu nhỏ 32x32 < 3 mức xám) là **chưa đủ**. Camera cố định và
   vật chỉ chiếm vài pixel, nên khoảng 2/3 số ảnh có nền "giống" một ảnh train dù vật đã đổi chỗ.
   Vì vậy một ảnh chỉ bị coi là trùng khi **nền gần trùng VÀ cùng các vật, cùng lớp, lệch ≤ 4 px**.
   Đó là các **frame liền nhau của cùng một buổi chụp** bị chia vào cả train lẫn val/test.

Đo trên dataset hiện tại: khoảng **21%** ảnh val/test là frame gần trùng với train. Các ảnh này được đánh dấu `near_dup`,
và mục 7 báo thêm điểm trên phần **`unseen`** (bỏ các ảnh gần trùng). Đây là con số **trung thực hơn** để đưa vào báo cáo.
Cách sửa triệt để là chia lại theo buổi chụp trên Roboflow, hoặc giữ riêng một buổi chụp mới làm test.

In [ ]:
for name, rows in (("val", M_val), ("test", M_test)):
    shared = set(rows["source_image"]) & set(M_train["source_image"])
    print(f"{name}: {len(shared)} source images also in train")


def thumbs(images: np.ndarray) -> np.ndarray:
    # 32x32 area-averaged thumbnails, flattened.
    small = [cv2.resize(i, (32, 32), interpolation=cv2.INTER_AREA) for i in images]
    return np.stack(small).reshape(len(images), -1).astype(np.float32)


def same_objects(a: list, b: list, tol_px: float = 4.0) -> bool:
    # True if both images hold the same objects (same classes, centres within tol_px).
    if len(a) != len(b):
        return False
    used = [False] * len(b)
    for cls, x, y, *_ in a:
        hit = next((j for j, t in enumerate(b)
                    if not used[j] and t[0] == cls and math.hypot(t[1] - x, t[2] - y) <= tol_px), None)
        if hit is None:
            return False
        used[hit] = True
    return True


def near_duplicates(images: np.ndarray, objects: list) -> np.ndarray:
    # Bool per image: some train frame has a near-identical background AND the same objects.
    flags = []
    for thumb, objs in zip(thumbs(images), objects):
        close = np.nonzero(np.abs(T_train - thumb).mean(axis=1) < 3)[0]
        flags.append(any(same_objects(objs, O_train[j]) for j in close))
    return np.array(flags)


T_train = thumbs(X_train)
for name, images, objects, rows in (("val", X_val, O_val, M_val), ("test", X_test, O_test, M_test)):
    nearest = np.array([np.abs(T_train - t).mean(axis=1).min() for t in thumbs(images)])
    rows["near_dup"] = near_duplicates(images, objects)
    print(f"{name}: background near-identical {np.mean(nearest < 3):.1%} (static camera, not leakage by itself) | "
          f"same background AND same objects: {rows['near_dup'].mean():.1%}  <- real leakage")

COLORS = {1: "red", 2: "lime"}


def show(images: np.ndarray, objects: list, preds: list | None = None, title: str = "", n: int = 16) -> None:
    # Grid of images; truth = circles, predictions = crosses, coloured by class.
    cols = 8
    count = min(n, len(images))
    rows = int(np.ceil(count / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(2.4 * cols, 2.4 * rows))
    for k, ax in enumerate(np.atleast_1d(axes).ravel()):
        ax.axis("off")
        if k >= count:
            continue
        ax.imshow(images[k], cmap="gray", vmin=0, vmax=255)
        for cls, cx, cy, *_ in objects[k]:
            ax.add_patch(plt.Circle((cx - 0.5, cy - 0.5), 5, fill=False, color=COLORS[cls], lw=1.5))
        for cls, cx, cy, *_ in (preds[k] if preds else []):
            ax.plot(cx - 0.5, cy - 0.5, "x", color=COLORS[cls], ms=8, mew=2)
    fig.suptitle(title + "  (o = nhãn, x = dự đoán; đỏ car, xanh lá human)")
    plt.tight_layout()
    plt.show()


rng_show = np.random.default_rng(SEED)
pick = rng_show.choice(len(X_train), min(16, len(X_train)), replace=False)
show(X_train[pick], [O_train[i] for i in pick], title="train")

## 3. Nhãn FOMO

Mỗi ảnh được đổi thành một lưới 20x20, mỗi ô là một vector one-hot 3 lớp (nền, car, human).
FOMO gốc chỉ đánh dấu **1 ô chứa tâm hộp**. Khi vật lớn hơn 1 ô, các ô khác cùng là thân vật lại bị gán "nền",
tức là nhãn mâu thuẫn (bản 96 của dự án đếm xe ra precision khoảng 0.5 vì lý do này). Vì vậy notebook dùng cách gán nhãn sau:
- **Lõi** = 1/3 giữa hộp theo mỗi chiều (tối đa 3 ô, luôn gồm ô tâm), mang lớp của vật.
- **Phần còn lại của hộp** được phép là "nền **hoặc** lớp của hộp", với loss `-log(p_nền + p_lớp)`.
  Nói nền hay nói đúng lớp đều không bị phạt, nhưng nói **lớp khác** thì bị phạt.
- Ngoài hộp: nền.

Vật ở đây nhỏ (khoảng 1.5 ô) nên lõi thường chỉ 1-2 ô, gần với FOMO gốc. Cách gán này chủ yếu có ích với xe ở gần (vật to).
Bộ giải mã (giống firmware) gộp các ô lõi kề nhau thành 1 vật và lấy tâm có trọng số.
Hình dưới: màu đậm = lõi, màu nhạt = "nền hoặc lớp này".

In [ ]:
CORE_FRACTION = 1.0 / 3.0          # core = middle third of the box (each axis) ...
CORE_MAX_HALF = 1.4 * CELL          # ... but at most 3 cells across
ALLOW_REST_OF_BOX = True            # cells inside the box but outside the core: "background OR the box's class"


def axis_cells(centre: float, half: float) -> range:
    # Cells whose centre lies within [centre - half, centre + half] (clipped to the grid).
    first = int(np.ceil((centre - half) / CELL - 0.5))
    last = int(np.floor((centre + half) / CELL - 0.5))
    return range(max(first, 0), min(last, GRID - 1) + 1)


def encode_targets(objects: list[tuple[int, float, float, float, float]]) -> np.ndarray:
    # FOMO target for one image: (GRID, GRID, len(CLASSES)), a set of allowed classes per cell.
    #
    # objects: (class index >= 1, centre x, centre y, box width, box height) in IMG_SIZE pixel coordinates.
    # The loss is -log(sum(target * prob)): a one-hot cell is plain cross-entropy, a cell allowing
    # {background, k} costs -log(p_background + p_k) - either answer is fine, any other class is punished.
    #   - core cells (middle third of the box, 1..3 cells per axis, always the centre cell) are one-hot class;
    #   - the rest of the box allows {background, the box's class} (several overlapping boxes: the union);
    #   - everything else is one-hot background.
    # Cores are written last, largest box first, so a small (distant) object's core is never hidden.
    target = np.zeros((GRID, GRID, len(CLASSES)), np.float32)
    target[..., 0] = 1.0
    ordered = sorted(objects, key=lambda o: -o[3] * o[4])
    if ALLOW_REST_OF_BOX:
        for cls, cx, cy, w, h in ordered:
            for row in axis_cells(cy, h / 2):
                for col in axis_cells(cx, w / 2):
                    target[row, col, cls] = 1.0
    for cls, cx, cy, w, h in ordered:
        col0, row0 = int(cx // CELL), int(cy // CELL)
        if not (0 <= col0 < GRID and 0 <= row0 < GRID):
            continue
        rows = set(axis_cells(cy, min(h * CORE_FRACTION / 2, CORE_MAX_HALF))) | {row0}
        cols = set(axis_cells(cx, min(w * CORE_FRACTION / 2, CORE_MAX_HALF))) | {col0}
        for row in rows:
            for col in cols:
                target[row, col, :] = 0.0
                target[row, col, cls] = 1.0
    return target


def allowed_cells(target: np.ndarray) -> np.ndarray:
    # (GRID, GRID) bool: cells that allow background and at least one class (rest of a box, outside its core).
    return (target[..., 0] == 1) & (target[..., 1:].sum(axis=-1) > 0)


def perfect_prediction(target: np.ndarray) -> np.ndarray:
    # What an ideal model would output for this target: cores keep their class, allowed cells say background.
    out = target.copy()
    mask = allowed_cells(target)
    out[mask] = 0.0
    out[mask, 0] = 1.0
    return out


Y_val = np.stack([encode_targets(o) for o in O_val])
from matplotlib.colors import to_rgb

object_cells = int((Y_val[..., 0] == 0).sum())
loose_cells = int(np.stack([allowed_cells(t) for t in Y_val]).sum())
n_objects = sum(len(o) for o in O_val)
print("val targets:", Y_val.shape, "| core cells:", object_cells,
      f"({object_cells / max(n_objects, 1):.2f} per object)",
      "| background-or-class cells:", loose_cells, "of", Y_val[..., 0].size)

fig, axes = plt.subplots(1, 8, figsize=(19, 2.4))
for ax, k in zip(axes, [i for i, o in enumerate(O_val) if o][:8]):
    overlay = np.stack([X_val[k]] * 3, axis=-1).astype(np.float32) / 255.0
    t = cv2.resize(Y_val[k], (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
    for cls in range(1, len(CLASSES)):
        colour = np.array(to_rgb(COLORS[cls]))
        loose = (t[..., cls] > 0) & (t[..., 0] > 0)
        core = (t[..., cls] > 0) & (t[..., 0] == 0)
        overlay[loose] = overlay[loose] * 0.75 + colour * 0.25                       # background or this class
        overlay[core] = overlay[core] * 0.3 + colour * 0.7                           # core: this class
    ax.imshow(overlay)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. Augmentation

Mỗi epoch, ảnh train được biến đổi ngẫu nhiên để model không học thuộc:
- **Hình học:** lật ngang, phóng/thu ±15%, dịch ±1/12 ảnh (khoảng 13 px ở 160).
- **Độ sáng:** ×0.45 đến ×1.25 để mô phỏng đèn **nền 20%** đến **100%** và lúc đang giảm sáng dở dang. Tương phản ±25%.
- **Nhiễu:** thêm nhiễu Gaussian nhẹ.

**Tâm vật được dời theo đúng phép biến đổi của ảnh**, nếu không nhãn sẽ lệch khỏi vật.
Không lật dọc và không xoay, vì góc camera cố định: người đứng thẳng, xe chạy trên mặt đường.

In [ ]:
def augment(image: np.ndarray, objects: list[tuple[int, float, float, float, float]],
            rng: np.random.Generator) -> tuple[np.ndarray, list[tuple[int, float, float, float, float]]]:
    # Random flip / scale / shift (box centres moved with the same matrix) + brightness, contrast, noise.
    # No vertical flip and no rotation: the camera is fixed and people / cars keep their orientation.
    scale = rng.uniform(0.85, 1.15)
    tx, ty = rng.uniform(-IMG_SIZE / 12, IMG_SIZE / 12, size=2)   # +-13 px at 160
    flip = rng.random() < 0.5
    centre = (IMG_SIZE - 1) / 2.0                       # pixel-index coordinates: pixel i is centred on i
    m = np.array([[scale, 0, (1 - scale) * centre + tx],
                  [0, scale, (1 - scale) * centre + ty],
                  [0, 0, 1]], np.float64)
    if flip:
        m = m @ np.array([[-1, 0, IMG_SIZE - 1], [0, 1, 0], [0, 0, 1]], np.float64)
    out = cv2.warpAffine(image, m[:2], (IMG_SIZE, IMG_SIZE), flags=cv2.INTER_LINEAR,
                         borderMode=cv2.BORDER_REPLICATE)
    moved = []
    for cls, cx, cy, w, h in objects:
        # COCO centres are continuous (pixel i spans [i, i+1)); shift by 0.5 into pixel-index space and back.
        x, y, _ = m @ np.array([cx - 0.5, cy - 0.5, 1.0])
        x, y = x + 0.5, y + 0.5
        if 0 <= x < IMG_SIZE and 0 <= y < IMG_SIZE:
            moved.append((cls, float(x), float(y), w * scale, h * scale))
    pixels = out.astype(np.float32)
    mean = pixels.mean()
    # brightness 0.45..1.25: lamps run between 20 % (base) and 100 %, plus fades in between
    pixels = (pixels - mean) * rng.uniform(0.75, 1.25) + mean * rng.uniform(0.45, 1.25)
    pixels += rng.normal(0.0, rng.uniform(0.0, 6.0), pixels.shape)
    return np.clip(pixels, 0, 255).astype(np.uint8), moved


def train_pairs(images: np.ndarray, objects: list, seed: int):
    # Endless generator of augmented (image, target) pairs.
    rng = np.random.default_rng(seed)
    while True:
        for i in rng.permutation(len(images)):
            image, moved = augment(images[i], objects[i], rng)
            yield image[..., None].astype(np.float32), encode_targets(moved)


signature = (tf.TensorSpec((IMG_SIZE, IMG_SIZE, 1), tf.float32),
             tf.TensorSpec((GRID, GRID, len(CLASSES)), tf.float32))
train_ds = (tf.data.Dataset.from_generator(lambda: train_pairs(X_train, O_train, SEED),
                                           output_signature=signature)
            .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))
STEPS_PER_EPOCH = int(np.ceil(len(X_train) / BATCH_SIZE))
val_inputs = X_val[..., None].astype(np.float32)

rng_demo = np.random.default_rng(1)
demo_index = next(i for i in pick if O_train[i])
demo = [augment(X_train[demo_index], O_train[demo_index], rng_demo) for _ in range(8)]
show(np.stack([d[0] for d in demo]), [d[1] for d in demo], title="8 biến thể của cùng một ảnh", n=8)

## 5. Model

**Đầu vào**: ảnh xám 0..255. Lớp `Rescaling` trong model đưa về [-1, 1], nên firmware chỉ cần đưa `pixel - 128`
(int8) vào. Kiểm tra lại `input.scale`/`zero_point` trong card ở mục 9.

**Backbone**: MobileNetV2 alpha 0.35 có trọng số ImageNet (3 kênh màu). Notebook cộng 3 kênh của lớp conv đầu thành 1 kênh.
Kết quả y hệt việc nhân ảnh xám ra 3 kênh nhưng model gọn hơn; dòng "max difference" in ra để chứng minh.
Cắt ở `block_6_expand_relu` (độ phân giải 1/8), nên ảnh 160 cho **lưới 20x20**, khớp `GRID` mà không cần đổi lưới.
Nếu đổi `CELL` sang cỡ không có trong MobileNetV2 (vd 10 px, lưới 16x16), model tự thêm khâu đổi lưới:
phóng nearest lên lưới chung, rồi trung bình về lưới đích.

**Đầu (head)**: conv 1x1 32 kênh, rồi conv 1x1 ra 3 lớp, rồi softmax theo từng ô. Conv 1x1 nghĩa là mỗi ô quyết
định dựa trên đặc trưng của chính vùng nó nhìn thấy, đúng tinh thần FOMO.

**RAM:** ở 160x160, activation lớn nhất nằm ở các lớp đầu (80x80x96 khoảng 600 KB int8), nên **tensor arena có thể vượt
512 KB SRAM nội** và cần đặt trong PSRAM. Ước lượng ở mục 9, con số chính xác phải đo trên board.

In [ ]:
def gray_backbone() -> keras.Model:
    # MobileNetV2 (ALPHA) on 1-channel input, ImageNet weights folded into the first conv, cut at CUT_LAYER.
    shape_rgb, shape_gray = (IMG_SIZE, IMG_SIZE, 3), (IMG_SIZE, IMG_SIZE, 1)
    try:
        rgb = keras.applications.MobileNetV2(input_shape=shape_rgb, alpha=ALPHA, include_top=False, weights="imagenet")
    except Exception as error:
        print("ImageNet weights unavailable (", error, ") -> training the backbone from scratch")
        rgb = keras.applications.MobileNetV2(input_shape=shape_rgb, alpha=ALPHA, include_top=False, weights=None)
    gray = keras.applications.MobileNetV2(input_shape=shape_gray, alpha=ALPHA, include_top=False, weights=None)
    for target, source in zip(gray.layers, rgb.layers):
        weights = source.get_weights()
        if target.name == "Conv1":           # first conv: sum the kernel over the R, G, B input channels
            weights = [weights[0].sum(axis=2, keepdims=True), *weights[1:]]
        target.set_weights(weights)
    probe = np.random.RandomState(0).uniform(-1, 1, (2, IMG_SIZE, IMG_SIZE, 1)).astype("float32")
    difference = np.abs(gray(probe).numpy() - rgb(np.repeat(probe, 3, axis=-1)).numpy()).max()
    print(f"gray vs RGB-replicated backbone, max difference: {difference:.2e}")
    return keras.Model(gray.input, gray.get_layer(CUT_LAYER).output, name="backbone")


def build_model() -> tuple[keras.Model, keras.Model]:
    # FOMO model (input grey 0..255 -> GRID x GRID x classes softmax) and its backbone (for freezing).
    backbone = gray_backbone()
    inputs = keras.Input((IMG_SIZE, IMG_SIZE, 1), name="gray")
    x = layers.Rescaling(1 / 127.5, offset=-1.0)(inputs)
    x = backbone(x)
    features = int(backbone.output_shape[1])
    if features != GRID:                # e.g. 20 -> 16: nearest x4 to the common grid 80, then average 5x5
        common = math.lcm(features, GRID)
        if common != features:
            x = layers.UpSampling2D(common // features, interpolation="nearest", name="to_common_grid")(x)
        x = layers.AveragePooling2D(common // GRID, name="to_output_grid")(x)
    x = layers.Conv2D(HEAD_FILTERS, 1, activation="relu", name="head")(x)
    outputs = layers.Conv2D(len(CLASSES), 1, activation="softmax", name="cells")(x)
    model = keras.Model(inputs, outputs, name=MODEL_STEM)
    if tuple(model.output_shape[1:3]) != (GRID, GRID):
        raise ValueError(f"output grid {model.output_shape} != {GRID}x{GRID}: check CUT_LAYER")
    return model, backbone


def weighted_cross_entropy(y_true: tf.Tensor, y_pred: tf.Tensor) -> tf.Tensor:
    # Per-cell -log(probability of the allowed classes); core cells weigh OBJECT_WEIGHT, the others 1.
    # y_true is one-hot (plain cross-entropy) or allows {background, k} (see encode_targets): then any mix of
    # background and k is equally fine, and only the other classes are punished.
    allowed_p = tf.reduce_sum(y_true * y_pred, axis=-1)
    ce = -tf.math.log(tf.clip_by_value(allowed_p, 1e-7, 1.0))
    weight = 1.0 + (OBJECT_WEIGHT - 1.0) * (1.0 - y_true[..., 0])
    return tf.reduce_mean(ce * weight)


model, backbone = build_model()
model.summary(line_length=100)

## 6. Giải mã, chấm điểm và huấn luyện

**Giải mã** (giống firmware):
1. Mỗi ô nhận lớp không-phải-nền có xác suất cao nhất, nếu xác suất đó ≥ ngưỡng.
2. Các ô **cùng lớp kề nhau (8 hướng)** được gộp thành 1 vật.
3. Tâm vật là trung bình có trọng số (theo xác suất) của tâm các ô; score là ô cao nhất.

**Chấm điểm:** dự đoán là TP nếu tâm cách tâm nhãn cùng lớp ≤ 12 px. Dự đoán thừa là FP, nhãn không được ghép là FN.
Hai nhãn cùng một ô chỉ tính là một, vì FOMO không thể tách chúng ra.

**Trần điểm:** trước khi train, chấm thử "model hoàn hảo" (đưa chính nhãn vào làm dự đoán). Điểm không đạt 1.0
vì **người đứng sát nhau** ở các ô kề nhau bị gộp thành 1 vật. Hãy so model với trần này chứ không phải với 1.0.
Với đèn đường thì không sao, vì 2 người sát nhau luôn cùng zone.

**Chọn model:** sau mỗi epoch, callback chấm tập **val** và giữ model có **macro F1 (car + human)** cao nhất,
vì cả hai lớp đều làm đèn bật. Không chọn theo `val_loss`, vì loss thấp chưa chắc tâm đã nằm đúng ô.

In [ ]:
def decode_heatmap(probs: np.ndarray, threshold: float) -> list[tuple[int, float, float, float]]:
    # FOMO output (GRID, GRID, len(CLASSES)) probabilities -> objects (class, centre x, centre y, score).
    # Same rule as the firmware:
    #   1. a cell belongs to its best non-background class when that class's probability >= threshold;
    #   2. 8-connected cells of the same class form one object;
    #   3. centre = probability-weighted mean of the cell centres (IMG_SIZE pixels), score = the highest cell.
    best = probs[..., 1:].argmax(axis=-1) + 1
    best_p = probs[..., 1:].max(axis=-1)
    label = np.where(best_p >= threshold, best, 0)
    seen = np.zeros(label.shape, bool)
    objects = []
    for row in range(GRID):
        for col in range(GRID):
            if label[row, col] == 0 or seen[row, col]:
                continue
            cls = int(label[row, col])
            stack, cells = [(row, col)], []
            seen[row, col] = True
            while stack:
                r, c = stack.pop()
                cells.append((r, c))
                for dr in (-1, 0, 1):
                    for dc in (-1, 0, 1):
                        rr, cc = r + dr, c + dc
                        if 0 <= rr < GRID and 0 <= cc < GRID and not seen[rr, cc] and label[rr, cc] == cls:
                            seen[rr, cc] = True
                            stack.append((rr, cc))
            weights = np.array([best_p[r, c] for r, c in cells])
            xs = np.array([(c + 0.5) * CELL for _, c in cells])
            ys = np.array([(r + 0.5) * CELL for r, _ in cells])
            objects.append((cls, float((xs * weights).sum() / weights.sum()),
                            float((ys * weights).sum() / weights.sum()), float(weights.max())))
    return objects


def match_counts(pred: list[tuple[int, float, float, float]], truth: list[tuple[int, float, float, float]],
                 radius_px: float = MATCH_RADIUS_PX) -> dict[int, list[int]]:
    # Per class [TP, FP, FN] for one image: greedy matching of centres within radius_px pixels.
    # Two truths in the same output cell can only ever be found once, so they count as one truth here.
    counts = {cls: [0, 0, 0] for cls in range(1, len(CLASSES))}
    for cls in counts:
        cells = {}
        for t in truth:
            if t[0] == cls:
                cells.setdefault((int(t[2] // CELL), int(t[1] // CELL)), t)
        gts = list(cells.values())
        preds = sorted([p for p in pred if p[0] == cls], key=lambda p: -p[3])
        used = [False] * len(gts)
        for p in preds:
            best, best_d = -1, radius_px
            for i, g in enumerate(gts):
                d = ((p[1] - g[1]) ** 2 + (p[2] - g[2]) ** 2) ** 0.5
                if not used[i] and d <= best_d:
                    best, best_d = i, d
            if best >= 0:
                used[best] = True
                counts[cls][0] += 1
            else:
                counts[cls][1] += 1
        counts[cls][2] += used.count(False)
    return counts


def score_table(all_probs: np.ndarray, all_truth: list, threshold: float) -> dict[str, dict[str, float]]:
    # Precision / recall / F1 per class over a set of images.
    total = {cls: [0, 0, 0] for cls in range(1, len(CLASSES))}
    for probs, truth in zip(all_probs, all_truth):
        for cls, (tp, fp, fn) in match_counts(decode_heatmap(probs, threshold), truth).items():
            total[cls][0] += tp
            total[cls][1] += fp
            total[cls][2] += fn
    table = {}
    for cls, (tp, fp, fn) in total.items():
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        table[CLASSES[cls]] = {"TP": tp, "FP": fp, "FN": fn, "precision": precision, "recall": recall, "f1": f1}
    return table


def macro_f1(table: dict[str, dict[str, float]]) -> float:
    return float(np.mean([v["f1"] for v in table.values()]))


def fp_breakdown(all_probs: np.ndarray, all_truth: list, threshold: float, cls: int,
                 radius_px: float = MATCH_RADIUS_PX) -> dict[str, int]:
    # Where the false positives of one class come from (same greedy matching as match_counts).
    # split       : centre inside a box of the same class whose truth was already matched -> one object
    #               decoded as several pieces;
    # other_class : centre inside a box of another class only -> class confusion (e.g. car on a person);
    # background  : centre inside no labelled box -> the model sees an object where nobody labelled one
    #               (a real false alarm, or a missing label).
    out = {"TP": 0, "split": 0, "other_class": 0, "background": 0}
    for probs, truth in zip(all_probs, all_truth):
        inside = lambda p, t: abs(p[1] - t[1]) <= t[3] / 2 and abs(p[2] - t[2]) <= t[4] / 2
        gts = [t for t in truth if t[0] == cls]
        used = [False] * len(gts)
        for p in sorted((p for p in decode_heatmap(probs, threshold) if p[0] == cls), key=lambda p: -p[3]):
            best, best_d = -1, radius_px
            for i, g in enumerate(gts):
                d = ((p[1] - g[1]) ** 2 + (p[2] - g[2]) ** 2) ** 0.5
                if not used[i] and d <= best_d:
                    best, best_d = i, d
            if best >= 0:
                used[best] = True
                out["TP"] += 1
            elif any(inside(p, t) for t in gts):
                out["split"] += 1
            elif any(inside(p, t) for t in truth):
                out["other_class"] += 1
            else:
                out["background"] += 1
    return out


ceiling = score_table(np.stack([perfect_prediction(t) for t in Y_val]), O_val, EVAL_THRESHOLD)
print("Ceiling (labels used as predictions, val):")
print(pd.DataFrame(ceiling).T.round(3))

In [ ]:
class ValF1(keras.callbacks.Callback):
    # Val macro F1 (firmware decoding) after each epoch; keeps the best weights; early stopping.

    def __init__(self) -> None:
        super().__init__()
        self.best_f1, self.best_weights, self.wait, self.history = -1.0, None, 0, []

    def on_epoch_end(self, epoch: int, logs: dict | None = None) -> None:
        probs = self.model.predict(val_inputs, verbose=0)
        table = score_table(probs, O_val, EVAL_THRESHOLD)
        macro = macro_f1(table)
        self.history.append({"epoch": len(self.history) + 1, "loss": (logs or {}).get("loss"),
                             "car_f1": table["car"]["f1"], "human_f1": table["human"]["f1"], "macro_f1": macro})
        print(f" - val macro F1 {macro:.3f} | car {table['car']['f1']:.3f} "
              f"(P {table['car']['precision']:.2f} R {table['car']['recall']:.2f}) | human {table['human']['f1']:.3f} "
              f"(P {table['human']['precision']:.2f} R {table['human']['recall']:.2f})")
        if macro > self.best_f1:
            self.best_f1, self.best_weights, self.wait = macro, self.model.get_weights(), 0
        else:
            self.wait += 1
            if self.wait >= PATIENCE:
                self.model.stop_training = True

    def restore(self) -> None:
        # Load the best weights seen so far.
        if self.best_weights is not None:
            self.model.set_weights(self.best_weights)


tracker = ValF1()
backbone.trainable = False
model.compile(optimizer=keras.optimizers.Adam(LEARNING_RATE), loss=weighted_cross_entropy)
print("phase 1: head only")
model.fit(train_ds, steps_per_epoch=STEPS_PER_EPOCH, epochs=EPOCHS_HEAD, callbacks=[tracker], verbose=2)
tracker.restore()
tracker.wait = 0
backbone.trainable = True
model.compile(optimizer=keras.optimizers.Adam(FINETUNE_LR), loss=weighted_cross_entropy)
print("phase 2: whole network")
model.fit(train_ds, steps_per_epoch=STEPS_PER_EPOCH, epochs=EPOCHS_FINETUNE, callbacks=[tracker], verbose=2)
tracker.restore()
print(f"best val macro F1: {tracker.best_f1:.3f}")

history = pd.DataFrame(tracker.history)
history.plot(x="epoch", y=["car_f1", "human_f1", "macro_f1"], grid=True, title="val F1 theo epoch")
plt.show()

## 7. Chọn ngưỡng trên val, đánh giá test một lần

Quét ngưỡng từ 0.3 đến 0.8 trên **val** và chọn ngưỡng có **macro F1** cao nhất. Giá trị này là `FOMO_SCORE_MIN` khởi điểm cho firmware.
Sau đó mới xem **test** với ngưỡng đó, tách thành các nhóm:
- `all`: toàn bộ ảnh test.
- `unseen`: bỏ các frame gần trùng train (mục 2). Đây là **con số nên báo cáo**.
- `320x240`: camera thật, con số quan trọng nhất.
- `640x360`: nguồn ảnh còn lại.

**Nguyên nhân FP** được tách thành 3 loại:
- `split`: 1 vật bị giải mã thành nhiều mảnh.
- `other_class`: nhầm người với xe.
- `background`: báo vật ở chỗ không có nhãn, tức là báo nhầm, hoặc là nhãn bị thiếu.

In [ ]:
THRESHOLDS = np.round(np.arange(0.30, 0.801, 0.05), 2)
p_val = model.predict(val_inputs, verbose=0)
sweep_rows = []
for t in THRESHOLDS:
    table = score_table(p_val, O_val, t)
    sweep_rows.append({"threshold": t, "macro_f1": macro_f1(table),
                       **{f"{c}_{k}": v[k] for c, v in table.items() for k in ("precision", "recall", "f1")}})
sweep = pd.DataFrame(sweep_rows)
print(sweep.round(3).to_string(index=False))
THRESHOLD = float(sweep.loc[sweep["macro_f1"].idxmax(), "threshold"])
print(f"\nchosen threshold (best val macro F1): {THRESHOLD:.2f}")


def report(probs: np.ndarray, objects: list, rows: pd.DataFrame, threshold: float) -> pd.DataFrame:
    # Per class P/R/F1 for all images, images without a near-duplicate in train, and each image source.
    out = {}
    groups = ([("all", np.ones(len(rows), bool)), ("unseen", ~rows["near_dup"].to_numpy())]
              + [(s, (rows["source"] == s).to_numpy()) for s in SOURCES])
    for name, mask in groups:
        if not mask.any():
            continue
        table = score_table(probs[mask], [o for o, m in zip(objects, mask) if m], threshold)
        for cls, v in table.items():
            out[(name, cls)] = v
    return pd.DataFrame(out).T


test_inputs = X_test[..., None].astype(np.float32)
p_test = model.predict(test_inputs, verbose=0)
test_float = report(p_test, O_test, M_test, THRESHOLD)
print("TEST, float model")
print(test_float.round(3))

rows = []
for source in SOURCES:
    mask = (M_test["source"] == source).to_numpy()
    for cls in range(1, len(CLASSES)):
        rows.append({"source": source, "class": CLASSES[cls],
                     **fp_breakdown(p_test[mask], [o for o, m in zip(O_test, mask) if m], THRESHOLD, cls)})
print("\nTEST false positives by cause")
print(pd.DataFrame(rows).to_string(index=False))


def false_positives(i: int) -> int:
    # Number of false positives (all classes) in test image i.
    total = 0
    for cls in range(1, len(CLASSES)):
        counts = fp_breakdown(p_test[i:i + 1], [O_test[i]], THRESHOLD, cls)
        total += counts["split"] + counts["other_class"] + counts["background"]
    return total


worst = sorted(range(len(X_test)), key=lambda i: -false_positives(i))[:8]
show(X_test[worst], [O_test[i] for i in worst], [decode_heatmap(p_test[i], THRESHOLD) for i in worst],
     title="test, most false positives", n=8)

## 8. Lượng tử int8 và so sánh với float

Firmware chạy model int8, vì model nhỏ hơn 4 lần và được ESP-NN tăng tốc. Lượng tử hoá luôn mất một chút độ chính xác.
Quy tắc của dự án: **chưa coi model là xong** khi chưa so int8 với float trên test và hiểu được chênh lệch.
Ảnh hiệu chỉnh lấy từ train, không bao giờ lấy val/test.

In [ ]:
rng_cal = np.random.default_rng(SEED)
CALIBRATION = X_train[rng_cal.permutation(len(X_train))[:CALIBRATION_IMAGES]]


def representative_dataset():
    # Calibration samples for the converter, one image at a time.
    for image in CALIBRATION:
        yield [image[None, ..., None].astype(np.float32)]


def convert_int8(keras_model: keras.Model) -> bytes:
    # Full-integer int8 TFLite flatbuffer with int8 input and output.
    converter = tf.lite.TFLiteConverter.from_keras_model(keras_model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = representative_dataset
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8
    return converter.convert()


def predict_tflite(flatbuffer: bytes, images: np.ndarray) -> np.ndarray:
    # Dequantised (N, GRID, GRID, classes) probabilities from the int8 model.
    interpreter = tf.lite.Interpreter(model_content=flatbuffer)
    interpreter.allocate_tensors()
    inp, out = interpreter.get_input_details()[0], interpreter.get_output_details()[0]
    (scale_in, zero_in), (scale_out, zero_out) = inp["quantization"], out["quantization"]
    result = []
    for image in images:
        q = np.clip(np.round(image.astype(np.float32) / scale_in + zero_in), -128, 127).astype(np.int8)
        interpreter.set_tensor(inp["index"], q[None, ..., None])
        interpreter.invoke()
        result.append((interpreter.get_tensor(out["index"])[0].astype(np.float32) - zero_out) * scale_out)
    return np.stack(result)


tflite_bytes = convert_int8(model)
print(f"int8 model: {len(tflite_bytes) / 1024:.1f} KB")
p_test_int8 = predict_tflite(tflite_bytes, X_test)
test_int8 = report(p_test_int8, O_test, M_test, THRESHOLD)
compare = pd.concat({"float": test_float[["precision", "recall", "f1"]],
                     "int8": test_int8[["precision", "recall", "f1"]]}, axis=1)
print("TEST, float vs int8")
print(compare.round(3))
print(f"mean |prob difference| float vs int8: {np.abs(p_test - p_test_int8).mean():.4f}")

### Xem dự đoán của model int8 trên test (ảnh 320x240 trước)

In [ ]:
order = np.argsort((M_test["source"] != "320x240").to_numpy(), kind="stable")[:64]
preds = [decode_heatmap(p_test_int8[i], THRESHOLD) for i in order]
show(X_test[order], [O_test[i] for i in order], preds, title="test, int8", n=64)

## 9. Xuất file cho firmware

- `fomo_160_g20.tflite`, `fomo_160_g20_data.h` (mảng C `alignas(16)` + `_len`): chép vào component vision của
  `firmware/` (2 file phải từ cùng một lần chạy).
- `fomo_160_g20_card.json` gồm:
  - tham số lượng tử, danh sách lớp, ngưỡng;
  - danh sách op, để khai báo `MicroMutableOpResolver`;
  - ước lượng arena. Đây là cận dưới; trên board thường lớn hơn 10-30%, cần đo lại bằng `arena_used_bytes()`.
    Nếu vượt khoảng 300 KB SRAM còn trống thì đặt arena trong PSRAM.

In [ ]:
NO_DELEGATE = tf.lite.experimental.OpResolverType.BUILTIN_WITHOUT_DEFAULT_DELEGATES


def estimate_arena_bytes(flatbuffer: bytes) -> int:
    # Lower bound of the tensor arena: peak total size of activation tensors alive at the same time.
    # No default delegate: XNNPACK would fold most ops into one DELEGATE node and hide their tensors.
    interpreter = tf.lite.Interpreter(model_content=flatbuffer, experimental_op_resolver_type=NO_DELEGATE)
    interpreter.allocate_tensors()
    details = {d["index"]: d for d in interpreter.get_tensor_details()}
    first, last = {}, {}
    for entry in interpreter.get_input_details():
        first[entry["index"]] = last[entry["index"]] = 0
    operations = interpreter._get_ops_details()
    for step, operation in enumerate(operations):
        for tensor in operation["outputs"]:
            first.setdefault(tensor, step)
            last[tensor] = max(last.get(tensor, step), step)
        for tensor in operation["inputs"]:
            if tensor in first:
                last[tensor] = max(last[tensor], step)

    def size(tensor: int) -> int:
        return int(np.prod(details[tensor]["shape"])) * np.dtype(details[tensor]["dtype"]).itemsize

    return max(sum(size(t) for t in first if first[t] <= step <= last[t]) for step in range(len(operations)))


def c_array(flatbuffer: bytes, name: str) -> str:
    # The model as a C array, 16-byte aligned, ready to include in firmware.
    rows = ["  " + ", ".join(f"0x{b:02x}" for b in flatbuffer[i:i + 16]) + ","
            for i in range(0, len(flatbuffer), 16)]
    return ("// Generated by train_fomo_160_g20.ipynb - do not edit by hand\n#pragma once\n"
            f"alignas(16) const unsigned char {name}[] = {{\n" + "\n".join(rows) +
            f"\n}};\nconst unsigned int {name}_len = {len(flatbuffer)};\n")


# Without the default XNNPACK delegate, so `ops` lists the real kernels (no "DELEGATE") for the firmware resolver.
interpreter = tf.lite.Interpreter(model_content=tflite_bytes, experimental_op_resolver_type=NO_DELEGATE)
interpreter.allocate_tensors()
inp, out = interpreter.get_input_details()[0], interpreter.get_output_details()[0]
ops = sorted({op["op_name"] for op in interpreter._get_ops_details()})
(OUT_DIR / f"{MODEL_STEM}.tflite").write_bytes(tflite_bytes)
(OUT_DIR / f"{MODEL_STEM}_data.h").write_text(c_array(tflite_bytes, f"{MODEL_STEM}_model"), encoding="utf-8")
card = {
    "created": datetime.datetime.now().isoformat(timespec="seconds"), "tensorflow": tf.__version__,
    "manifest_hash": MANIFEST_HASH, "quick_run": QUICK_RUN, "md5": hashlib.md5(tflite_bytes).hexdigest(),
    "classes": CLASSES, "grid": GRID, "cell_px": CELL,
    "input": {"shape": [int(v) for v in inp["shape"]], "scale": float(inp["quantization"][0]),
              "zero_point": int(inp["quantization"][1]),
              "meaning": "grey pixel 0..255 -> int8 = round(pixel / scale + zero_point)"},
    "output": {"shape": [int(v) for v in out["shape"]], "scale": float(out["quantization"][0]),
               "zero_point": int(out["quantization"][1]),
               "meaning": "probability = (q - zero_point) * scale, softmax over classes per cell"},
    "preprocess": f"grey frame (same ROI as the dataset) -> squash whole frame to {IMG_SIZE}x{IMG_SIZE} "
                  "with area averaging (cv2.INTER_AREA), no crop, no equalisation",
    "dataset": str(COCO_ROOT.name),
    "decode": "cell -> best non-background class if prob >= threshold; 8-connected same-class cells = 1 object; "
              "centre = prob-weighted mean of cell centres; score = max cell prob",
    "zones": f"zone = floor(centre_x * 4 / {IMG_SIZE}) for 4 lamps along x",
    "threshold": THRESHOLD, "ops": ops,
    "size_kb": round(len(tflite_bytes) / 1024, 1),
    "arena_lower_bound_kb": round(estimate_arena_bytes(tflite_bytes) / 1024, 1),
    "test_int8": {f"{d}/{c}": {k: round(float(v), 3) for k, v in row.items()} for (d, c), row in test_int8.iterrows()},
}
(OUT_DIR / f"{MODEL_STEM}_card.json").write_text(json.dumps(card, indent=2), encoding="utf-8")
history.to_csv(OUT_DIR / "history.csv", index=False)
model.save(OUT_DIR / f"{MODEL_STEM}_float.keras")
print(json.dumps({k: card[k] for k in ("size_kb", "arena_lower_bound_kb", "threshold", "ops", "input", "output")},
                 indent=2))
archive = shutil.make_archive(str(WORK / f"result_{MODEL_STEM}"), "zip", OUT_DIR)
print("result:", archive)
if "google.colab" in sys.modules:
    from google.colab import files
    files.download(archive)

## 10. Bước tiếp theo

1. Tải `result_fomo_160_g20.zip`. Chép `fomo_160_g20.tflite`, `fomo_160_g20_data.h` và `fomo_160_g20_card.json` vào
   component vision của firmware. Kiểm tra danh sách `ops` trong card có khớp với `MicroMutableOpResolver` không.
2. **Tiền xử lý trên board phải giống notebook:** frame GRAYSCALE, cắt ROI giống lúc thu dataset, ép về 160x160 bằng
   trung bình diện tích, rồi `int8 = pixel + zero_point` (scale = 1).
3. **Hậu xử lý:** giải mã giống `decode_heatmap`, rồi `zone = centre_x * 4 / 160`. `threshold` trong card là giá trị
   `FOMO_SCORE_MIN` khởi điểm; chỉnh tiếp theo cảnh thật (đèn 20%).
4. **Đo trên board:** `arena_used_bytes()`, thời gian `Invoke()` (mục tiêu ≥ 3-5 lần/giây), `idf.py size-components`.
5. Nếu điểm của nguồn **320x240** thấp hơn hẳn: cần thêm ảnh từ chính camera demo, đặc biệt ảnh tối ở đèn 20%, vật ở ranh giới zone, và người đứng sát nhau.